In [3]:
data_dir = 'D:/Student/anaconda3/envs/CC1/RootDir/data/hmnist_28_28_RGB.csv'
data = pd.read_csv(data_dir)
data.head()

,pixel0000,pixel0001,pixel0002,pixel0003,pixel0004,pixel0005,pixel0006,pixel0007,pixel0008,pixel0009,...,pixel2343,pixel2344,pixel2345,pixel2346,pixel2347,pixel2348,pixel2349,pixel2350,pixel2351,label
0,192,153,193,195,155,192,197,154,185,202,...,173,124,138,183,147,166,185,154,177,2
1,25,14,30,68,48,75,123,93,126,158,...,60,39,55,25,14,28,25,14,27,2
2,192,138,153,200,145,163,201,142,160,206,...,167,129,143,159,124,142,136,104,117,2
3,38,19,30,95,59,72,143,103,119,171,...,44,26,36,25,12,17,25,12,15,2
4,158,113,139,194,144,174,215,162,191,225,...,209,166,185,172,135,149,109,78,92,2


In [4]:
import numpy as np
import pandas as pd
import os
from glob import glob
import seaborn as sns
from PIL import Image
np.random.seed(123)
import itertools
import keras
from keras.utils import to_categorical 
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPool2D
from keras import backend as K
from keras.layers import BatchNormalization
from keras.utils import to_categorical 
from keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.callbacks import ReduceLROnPlateau
%matplotlib inline
import matplotlib.pyplot as plt

In [5]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

In [6]:
base_skin_dir = os.path.join('..', 'input')

# Merging images from both folders HAM10000_images_part1.zip and HAM10000_images_part2.zip into one dictionary

data_dir = 'D:/Student/anaconda3/envs/CC1/RootDir/data'
all_image_path = glob(os.path.join(data_dir, '*', '*.jpg'))
imageid_path_dict = {os.path.splitext(os.path.basename(x))[0]: x for x in all_image_path}
lesion_type_dict = {
    'nv': 'Melanocytic nevi',
    'mel': 'dermatofibroma',
    'bkl': 'Benign keratosis-like lesions ',
    'bcc': 'Basal cell carcinoma',
    'akiec': 'Actinic keratoses',
    'vasc': 'Vascular lesions',
    'df': 'Dermatofibroma'
}


In [7]:
skin_df = pd.read_csv(os.path.join(base_skin_dir, 'D:/Student/anaconda3/envs/CC1/RootDir/data/HAM10000_metadata.csv'))

# Creating New Columns for better readability

skin_df['path'] = skin_df['image_id'].map(imageid_path_dict.get)
skin_df['cell_type'] = skin_df['dx'].map(lesion_type_dict.get) 
skin_df['cell_type_idx'] = pd.Categorical(skin_df['cell_type']).codes

# Now lets see the sample of tile_df to look on newly made columns
skin_df.head(1000)

,lesion_id,image_id,dx,dx_type,age,sex,localization,path,cell_type,cell_type_idx
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,D:/Student/anaconda3/envs/CC1/RootDir/data\HAM...,Benign keratosis-like lesions,2
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,D:/Student/anaconda3/envs/CC1/RootDir/data\HAM...,Benign keratosis-like lesions,2
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,D:/Student/anaconda3/envs/CC1/RootDir/data\HAM...,Benign keratosis-like lesions,2
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,D:/Student/anaconda3/envs/CC1/RootDir/data\HAM...,Benign keratosis-like lesions,2
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,D:/Student/anaconda3/envs/CC1/RootDir/data\HAM...,Benign keratosis-like lesions,2
...,...,...,...,...,...,...,...,...,...,...
995,HAM_0000734,ISIC_0031196,bkl,consensus,45.0,male,abdomen,D:/Student/anaconda3/envs/CC1/RootDir/data\HAM...,Benign keratosis-like lesions,2
996,HAM_0005834,ISIC_0032249,bkl,consensus,45.0,male,lower extremity,D:/Student/anaconda3/envs/CC1/RootDir/data\HAM...,Benign keratosis-like lesions,2
997,HAM_0005136,ISIC_0024337,bkl,consensus,50.0,female,lower extremity,D:/Student/anaconda3/envs/CC1/RootDir/data\HAM...,Benign keratosis-like lesions,2
998,HAM_0004308,ISIC_0027370,bkl,consensus,50.0,female,back,D:/Student/anaconda3/envs/CC1/RootDir/data\HAM...,Benign keratosis-like lesions,2


In [8]:
skin_df['image']=skin_df['path'].map(lambda x: np.asarray(Image.open(x).resize((100,75))))

In [9]:
n_samples =5 # number of samples of each cancer type

# Plotting code
fig, m_axs = plt.subplots(7, n_samples, figsize = (4*n_samples, 3*7))
for n_axs, (type_name, type_rows) in zip(m_axs, 
                                         skin_df.sort_values(['cell_type']).groupby('cell_type')):
    n_axs[0].set_title(type_name)
    for c_ax, (_, c_row) in zip(n_axs, type_rows.sample(n_samples, random_state=1234).iterrows()):
        c_ax.imshow(c_row['image'])
        c_ax.axis('off')
fig.savefig('category_samples.png', dpi=300)

In [10]:
target=skin_df['cell_type_idx']
features = skin_df.drop(columns=['cell_type_idx'], axis=1)

In [11]:
x_train_o, x_test_o, y_train_o, y_test_o = train_test_split(features, target, test_size=0.20,random_state=123)


In [12]:
x_train = np.asarray(x_train_o['image'].tolist())
x_test = np.asarray(x_test_o['image'].tolist())

x_train_mean = np.mean(x_train)
x_train_std = np.std(x_train)

x_test_mean = np.mean(x_test)
x_test_std = np.std(x_test)

x_train = (x_train - x_train_mean)/x_train_std
x_test = (x_test - x_test_mean)/x_test_std

In [13]:
# Perform one-hot encoding on the labels
y_train = to_categorical(y_train_o, num_classes = 7)
y_test = to_categorical(y_test_o, num_classes = 7)

In [14]:
x_train, x_validate, y_train, y_validate = train_test_split(x_train, y_train, test_size = 0.1, random_state = 2)


In [15]:
# Reshape image in 3 dimensions (height = 75px, width = 100px , canal = 3)
x_train = x_train.reshape(x_train.shape[0], *(75, 100, 3))
x_test = x_test.reshape(x_test.shape[0], *(75, 100, 3))
x_validate = x_validate.reshape(x_validate.shape[0], *(75, 100, 3))

In [16]:
input_shape = (75, 100, 3)
num_classes = 7

model = Sequential()
model.add(Conv2D(32, kernel_size=(3, 3),activation='relu',padding = 'Same',input_shape=input_shape))
model.add(Conv2D(32,kernel_size=(3, 3), activation='relu',padding = 'Same',))
model.add(MaxPool2D(pool_size = (2, 2)))
model.add(Dropout(0.25))

model.add(Conv2D(64, (3, 3), activation='relu',padding = 'Same'))
model.add(Conv2D(64, (3, 3), activation='relu',padding = 'Same'))
model.add(MaxPool2D(pool_size=(2, 2)))
model.add(Dropout(0.40))

model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(num_classes, activation='softmax'))
model.summary()

D:\Student\anaconda3\envs\CC1\lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 75, 100, 32)         │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 75, 100, 32)         │           9,248 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 37, 50, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 37, 50, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 37, 50, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 37, 50, 64)          │          36,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 18, 25, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 18, 25, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 28800)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │       3,686,528 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 7)                   │             903 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,752,999 (14.32 MB)

 Trainable params: 3,752,999 (14.32 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
# Define the optimizer
optimizer = keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999, decay=0.0, amsgrad=False)

# Compile the model
model.compile(optimizer = optimizer , loss = "categorical_crossentropy", metrics=["accuracy"])

# Set a learning rate annealer
learning_rate_reduction = ReduceLROnPlateau(monitor='val_acc', 
                                            patience=3, 
                                            verbose=1, 
                                            factor=0.5, 
                                            min_lr=0.00001)

D:\Student\anaconda3\envs\CC1\lib\site-packages\keras\src\optimizers\base_optimizer.py:33: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


In [18]:
# With data augmentation to prevent overfitting 

datagen = ImageDataGenerator(
        featurewise_center=False,  # set input mean to 0 over the dataset
        samplewise_center=False,  # set each sample mean to 0
        featurewise_std_normalization=False,  # divide inputs by std of the dataset
        samplewise_std_normalization=False,  # divide each input by its std
        zca_whitening=False,  # apply ZCA whitening
        rotation_range=10,  # randomly rotate images in the range (degrees, 0 to 180)
        zoom_range = 0.1, # Randomly zoom image 
        width_shift_range=0.1,  # randomly shift images horizontally (fraction of total width)
        height_shift_range=0.1,  # randomly shift images vertically (fraction of total height)
        horizontal_flip=False,  # randomly flip images
        vertical_flip=False)  # randomly flip images

datagen.fit(x_train)

In [19]:
epochs = 100
batch_size = 10
test_generator =datagen.flow(x_train, y_train, batch_size=batch_size)
history = model.fit(test_generator,
                    epochs=epochs,
                    validation_data=(x_validate, y_validate),
                    verbose=1,
                    steps_per_epoch=x_train.shape[0] // batch_size,
                    callbacks=[learning_rate_reduction])

Epoch 1/100


D:\Student\anaconda3\envs\CC1\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


721/721 ━━━━━━━━━━━━━━━━━━━━ 41s 54ms/step - accuracy: 0.6547 - loss: 1.0876 - val_accuracy: 0.6633 - val_loss: 0.9114 - learning_rate: 0.0010
Epoch 2/100


D:\Student\anaconda3\envs\CC1\lib\site-packages\keras\src\callbacks\callback_list.py:96: UserWarning: Learning rate reduction is conditioned on metric `val_acc` which is not available. Available metrics are: accuracy,loss,val_accuracy,val_loss,learning_rate.
  callback.on_epoch_end(epoch, logs)
D:\Student\anaconda3\envs\CC1\lib\contextlib.py:153: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(typ, value, traceback)


721/721 ━━━━━━━━━━━━━━━━━━━━ 1s 889us/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.6633 - val_loss: 0.9114 - learning_rate: 0.0010
Epoch 3/100
721/721 ━━━━━━━━━━━━━━━━━━━━ 39s 54ms/step - accuracy: 0.6846 - loss: 0.9134 - val_accuracy: 0.6559 - val_loss: 0.8909 - learning_rate: 0.0010
Epoch 4/100
721/721 ━━━━━━━━━━━━━━━━━━━━ 1s 832us/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.6559 - val_loss: 0.8909 - learning_rate: 0.0010
Epoch 5/100
721/721 ━━━━━━━━━━━━━━━━━━━━ 39s 54ms/step - accuracy: 0.6812 - loss: 0.8949 - val_accuracy: 0.6721 - val_loss: 0.8588 - learning_rate: 0.0010
Epoch 6/100
721/721 ━━━━━━━━━━━━━━━━━━━━ 1s 850us/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.6721 - val_loss: 0.8588 - learning_rate: 0.0010
Epoch 7/100
721/721 ━━━━━━━━━━━━━━━━━━━━ 39s 54ms/step - accuracy: 0.6865 - loss: 0.8742 - val_accuracy: 0.6783 - val_loss: 0.8499 - learning_rate: 0.0010
Epoch 8/100
721/721 ━━━━━━━━━━━━━━━━━━━━ 1s 854us/step - a

In [20]:
loss, accuracy = model.evaluate(x_test, y_test, verbose=1)
loss_v, accuracy_v = model.evaluate(x_validate, y_validate, verbose=1)
print("Validation: accuracy = %f  ;  loss_v = %f" % (accuracy_v, loss_v))
print("Test: accuracy = %f  ;  loss = %f" % (accuracy, loss))
model.save("modelMulti100.h5")

63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.7614 - loss: 0.6759
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.7609 - loss: 0.6867


Validation: accuracy = 0.759352  ;  loss_v = 0.686153
Test: accuracy = 0.746880  ;  loss = 0.692610


In [28]:
input_shape = (75, 100, 3)
num_classes = 7

model2 = Sequential()
model2.add(Conv2D(32, kernel_size=(3, 3),activation='relu',padding = 'Same',input_shape=input_shape))
model2.add(Conv2D(32,kernel_size=(3, 3), activation='relu',padding = 'Same',))
model2.add(MaxPool2D(pool_size = (2, 2)))
model2.add(Dropout(0.25))

model2.add(Conv2D(64, (3, 3), activation='relu',padding = 'Same'))
model2.add(Conv2D(64, (3, 3), activation='relu',padding = 'Same'))
model2.add(MaxPool2D(pool_size=(2, 2)))
model2.add(Dropout(0.40))

model2.add(Flatten())
model2.add(Dense(128, activation='relu'))
model2.add(Dropout(0.5))
model2.add(Dense(num_classes, activation='softmax'))
model2.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_8 (Conv2D)                    │ (None, 75, 100, 32)         │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_9 (Conv2D)                    │ (None, 75, 100, 32)         │           9,248 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_4 (MaxPooling2D)       │ (None, 37, 50, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_8 (Dropout)                  │ (None, 37, 50, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_10 (Conv2D)                   │ (None, 37, 50, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_11 (Conv2D)                   │ (None, 37, 50, 64)          │          36,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_5 (MaxPooling2D)       │ (None, 18, 25, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_9 (Dropout)                  │ (None, 18, 25, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_3 (Flatten)                  │ (None, 28800)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 128)                 │       3,686,528 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_10 (Dropout)                 │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_8 (Dense)                      │ (None, 7)                   │             903 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,752,999 (14.32 MB)

 Trainable params: 3,752,999 (14.32 MB)

 Non-trainable params: 0 (0.00 B)

In [29]:
from keras.optimizers import Lion
opt2 = keras.optimizers.Lion(
    learning_rate=0.001,
    beta_1=0.9,
    beta_2=0.999,
    weight_decay=None,
    clipnorm=None,
    clipvalue=None,
    global_clipnorm=None,
    use_ema=False,
    ema_momentum=0.99,
    ema_overwrite_frequency=None,
    loss_scale_factor=None,
    gradient_accumulation_steps=None,
    name="lion")


In [30]:
# Compile the model
model2.compile(optimizer = opt2 , loss = "categorical_crossentropy", metrics=["accuracy"])

# Set a learning rate annealer
learning_rate_reduction = ReduceLROnPlateau(monitor='val_acc', 
                                            patience=3, 
                                            verbose=1, 
                                            factor=0.5, 
                                            min_lr=0.00001)

In [31]:
datagen = ImageDataGenerator(
        featurewise_center=False,  # set input mean to 0 over the dataset
        samplewise_center=False,  # set each sample mean to 0
        featurewise_std_normalization=False,  # divide inputs by std of the dataset
        samplewise_std_normalization=False,  # divide each input by its std
        zca_whitening=False,  # apply ZCA whitening
        rotation_range=10,  # randomly rotate images in the range (degrees, 0 to 180)
        zoom_range = 0.1, # Randomly zoom image 
        width_shift_range=0.1,  # randomly shift images horizontally (fraction of total width)
        height_shift_range=0.1,  # randomly shift images vertically (fraction of total height)
        horizontal_flip=False,  # randomly flip images
        vertical_flip=False)  # randomly flip images

datagen.fit(x_train)

In [32]:
epochs = 50 
batch_size = 10
test_generator =datagen.flow(x_train, y_train, batch_size=batch_size)
history = model2.fit(test_generator,
                    epochs=epochs,
                    validation_data=(x_validate, y_validate),
                    verbose=1,
                    steps_per_epoch=x_train.shape[0] // batch_size,
                    callbacks=[learning_rate_reduction])

Epoch 1/50
721/721 ━━━━━━━━━━━━━━━━━━━━ 38s 50ms/step - accuracy: 0.6491 - loss: 1.1891 - val_accuracy: 0.6546 - val_loss: 1.0913 - learning_rate: 0.0010
Epoch 2/50
721/721 ━━━━━━━━━━━━━━━━━━━━ 1s 938us/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.6546 - val_loss: 1.0913 - learning_rate: 0.0010
Epoch 3/50
721/721 ━━━━━━━━━━━━━━━━━━━━ 36s 50ms/step - accuracy: 0.6687 - loss: 1.1319 - val_accuracy: 0.6546 - val_loss: 1.1537 - learning_rate: 0.0010
Epoch 4/50
721/721 ━━━━━━━━━━━━━━━━━━━━ 1s 885us/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.6546 - val_loss: 1.1537 - learning_rate: 0.0010
Epoch 5/50
721/721 ━━━━━━━━━━━━━━━━━━━━ 35s 48ms/step - accuracy: 0.6717 - loss: 1.1526 - val_accuracy: 0.6546 - val_loss: 1.1418 - learning_rate: 0.0010
Epoch 6/50
721/721 ━━━━━━━━━━━━━━━━━━━━ 1s 829us/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.6546 - val_loss: 1.1418 - learning_rate: 0.0010
Epoch 7/50
721/721 ━━━━━━━━━━━━━━━━━━━━ 35s 48ms/ste

In [33]:
loss, accuracy = model2.evaluate(x_test, y_test, verbose=1)
loss_v, accuracy_v = model2.evaluate(x_validate, y_validate, verbose=1)
print("Validation: accuracy = %f  ;  loss_v = %f" % (accuracy_v, loss_v))
print("Test: accuracy = %f  ;  loss = %f" % (accuracy, loss))
model2.save("modelLion50.h5")

63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.6618 - loss: 1.1479
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.6513 - loss: 1.1714


Validation: accuracy = 0.654613  ;  loss_v = 1.146993
Test: accuracy = 0.659011  ;  loss = 1.153434


In [21]:
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense
from tensorflow.keras.models import Model
from tensorflow.keras import Input

# Define the input shape
input_shape = (75, 100, 3)
num_classes = 7

# Define the model architecture
inputs = Input(shape=input_shape)
x = Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same')(inputs)
x = Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same')(x)
x = MaxPooling2D(pool_size=(2, 2))(x)
x = Dropout(0.25)(x)

x = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(x)
x = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(x)
x = MaxPooling2D(pool_size=(2, 2))(x)
x = Dropout(0.4)(x)

x = Flatten()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
outputs = Dense(num_classes, activation='sigmoid')(x)

model3 = Model(inputs=inputs, outputs=outputs)
model3.summary()

Model: "functional_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)           │ (None, 75, 100, 3)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_4 (Conv2D)                    │ (None, 75, 100, 32)         │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_5 (Conv2D)                    │ (None, 75, 100, 32)         │           9,248 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 37, 50, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 37, 50, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_6 (Conv2D)                    │ (None, 37, 50, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_7 (Conv2D)                    │ (None, 37, 50, 64)          │          36,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 18, 25, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_4 (Dropout)                  │ (None, 18, 25, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_1 (Flatten)                  │ (None, 28800)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 128)                 │       3,686,528 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_5 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 7)                   │             903 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,752,999 (14.32 MB)

 Trainable params: 3,752,999 (14.32 MB)

 Non-trainable params: 0 (0.00 B)

In [22]:
# Define the optimizer
optimizer = keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999, decay=0.0, amsgrad=False)

# Compile the model
model3.compile(optimizer = optimizer , loss = "binary_crossentropy", metrics=["accuracy"])

# Set a learning rate annealer
learning_rate_reduction = ReduceLROnPlateau(monitor='val_acc', 
                                            patience=3, 
                                            verbose=1, 
                                            factor=0.5, 
                                            min_lr=0.00001)

In [23]:
epochs = 100 
batch_size = 10

test_generator =datagen.flow(x_train, y_train, batch_size=batch_size)
history = model3.fit(test_generator,
                    epochs=epochs,
                    validation_data=(x_validate, y_validate),
                    verbose=1,
                    steps_per_epoch=x_train.shape[0] // batch_size,
                    callbacks=[learning_rate_reduction])

Epoch 1/100
721/721 ━━━━━━━━━━━━━━━━━━━━ 42s 55ms/step - accuracy: 0.6530 - loss: 0.2731 - val_accuracy: 0.6584 - val_loss: 0.2092 - learning_rate: 0.0010
Epoch 2/100
721/721 ━━━━━━━━━━━━━━━━━━━━ 1s 971us/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.6584 - val_loss: 0.2092 - learning_rate: 0.0010
Epoch 3/100
721/721 ━━━━━━━━━━━━━━━━━━━━ 39s 54ms/step - accuracy: 0.6727 - loss: 0.2205 - val_accuracy: 0.6833 - val_loss: 0.2113 - learning_rate: 0.0010
Epoch 4/100
721/721 ━━━━━━━━━━━━━━━━━━━━ 1s 856us/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.6833 - val_loss: 0.2113 - learning_rate: 0.0010
Epoch 5/100
721/721 ━━━━━━━━━━━━━━━━━━━━ 39s 54ms/step - accuracy: 0.6764 - loss: 0.2100 - val_accuracy: 0.6696 - val_loss: 0.2085 - learning_rate: 0.0010
Epoch 6/100
721/721 ━━━━━━━━━━━━━━━━━━━━ 1s 857us/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.6696 - val_loss: 0.2085 - learning_rate: 0.0010
Epoch 7/100
721/721 ━━━━━━━━━━━━━━━━━━━━ 40s 5

In [24]:
loss, accuracy = model3.evaluate(x_test, y_test, verbose=1)
loss_v, accuracy_v = model3.evaluate(x_validate, y_validate, verbose=1)
print("Validation: accuracy = %f  ;  loss_v = %f" % (accuracy_v, loss_v))
print("Test: accuracy = %f  ;  loss = %f" % (accuracy, loss))
model3.save("modelBinary100.h5")

63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.7655 - loss: 0.1552
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.7523 - loss: 0.1527


Validation: accuracy = 0.750623  ;  loss_v = 0.154248
Test: accuracy = 0.761857  ;  loss = 0.158375


In [25]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Flatten, Dense, Dropout, concatenate

input_shape = (75, 100, 3)
input_layer = Input(shape=input_shape)

flatten = Flatten()(input_layer)
dense1 = Dense(256, activation='relu')(flatten)
dropout1 = Dropout(0.5)(dense1)
dense2 = Dense(128, activation='relu')(dropout1)
dropout2 = Dropout(0.5)(dense2)
output = Dense(num_classes, activation='softmax')(dropout2)
model4 = Model(inputs=input_layer, outputs=output)
model4.summary()

Model: "functional_13"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)           │ (None, 75, 100, 3)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_2 (Flatten)                  │ (None, 22500)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 256)                 │       5,760,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_6 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 128)                 │          32,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_7 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 7)                   │             903 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 5,794,055 (22.10 MB)

 Trainable params: 5,794,055 (22.10 MB)

 Non-trainable params: 0 (0.00 B)

In [26]:
# Compile and train the model
optimizer = Adam(learning_rate=0.001)

model4.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

model4.fit(x_train, y_train, batch_size=10, epochs=50, validation_data=(x_validate, y_validate),steps_per_epoch=x_train.shape[0] // 10,
                    callbacks=[learning_rate_reduction])


Epoch 1/50
721/721 ━━━━━━━━━━━━━━━━━━━━ 24s 32ms/step - accuracy: 0.5136 - loss: 9.4613 - val_accuracy: 0.6571 - val_loss: 1.6010 - learning_rate: 0.0010
Epoch 2/50
721/721 ━━━━━━━━━━━━━━━━━━━━ 0s 343us/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.6571 - val_loss: 1.6010 - learning_rate: 0.0010
Epoch 3/50
721/721 ━━━━━━━━━━━━━━━━━━━━ 22s 31ms/step - accuracy: 0.6654 - loss: 1.8080 - val_accuracy: 0.6559 - val_loss: 1.3729 - learning_rate: 0.0010
Epoch 4/50
721/721 ━━━━━━━━━━━━━━━━━━━━ 0s 322us/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.6559 - val_loss: 1.3729 - learning_rate: 0.0010
Epoch 5/50
721/721 ━━━━━━━━━━━━━━━━━━━━ 22s 31ms/step - accuracy: 0.6559 - loss: 1.7171 - val_accuracy: 0.6596 - val_loss: 1.1777 - learning_rate: 0.0010
Epoch 6/50
721/721 ━━━━━━━━━━━━━━━━━━━━ 0s 340us/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.6596 - val_loss: 1.1777 - learning_rate: 0.0010
Epoch 7/50
721/721 ━━━━━━━━━━━━━━━━━━━━ 23s 31ms/ste

In [27]:
# Evaluate the model4
loss, accuracy = model4.evaluate(x_test, y_test, verbose=1)
loss_v, accuracy_v = model4.evaluate(x_validate, y_validate, verbose=1)
print("Validation: accuracy = %f  ;  loss_v = %f" % (accuracy_v, loss_v))
print("Test: accuracy = %f  ;  loss = %f" % (accuracy, loss))
model4.save("modelNeural50.h5")

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6623 - loss: 1.1277
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6513 - loss: 1.1515


Validation: accuracy = 0.654613  ;  loss_v = 1.132436
Test: accuracy = 0.659511  ;  loss = 1.135215


In [39]:
import tensorflow as tf
import pandas as pd
from PIL import Image
import numpy as np

# تحميل البيانات من ملف CSV
data = pd.read_csv('/kaggle/input/test-1/test.csv')
print(data.values)
# تحميل النموذج من الملف H5
model = tf.keras.models.load_model('/kaggle/working/model.h5')

# تحميل الصورة وتحويلها إلى الحجم المطلوب
image = Image.open('/kaggle/input/test-1/download (1).jfif')
image = image.resize((100, 75))

# تحويل الصورة إلى مصفوفة NumPy وتغيير حجمها
image_array = np.array(image)
image_array = image_array.reshape(1, 75, 100, 3)
image_array = image_array.astype('float32') / 255.0  # تقليل قيم البكسل لتتراوح بين 0 و1

# تحويل المصفوفة إلى Tensor
image_tensor = tf.convert_to_tensor(image_array)

# دمج البيانات والصورة معًا
combined_input = [data.values, image_tensor]

# تنفيذ التنبؤ باستخدام النموذج
predictions = model.predict(combined_input)

# تحويل النتائج إلى تنبؤ فئة أو قيمة محددة
predicted_class = np.argmax(predictions)

print('Predicted Class:', predicted_class, predictions)

[['HAM_0000118' 'download (1)' 'histo' 65 'male' 'hand']]


ValueError: Failed to convert a NumPy array to a Tensor (Unsupported object type int).

In [41]:
def predict_output(image_path, data_path):
    # تحميل الصورة
    image = Image.open(image_path)
    image = image.resize((100, 75))  # تغيير الحجم إلى الحجم المطلوب
    image = np.asarray(image)
    image = image.reshape(1, *(75, 100, 3))
    
    # قراءة بيانات المريض من ملف البيانات
    patient_data = pd.read_csv(data_path)
    
    # تنسيق البيانات كملف داتا
    data = np.array(patient_data)  # قم بتنسيق البيانات بطريقة مناسبة هنا
    
    # تنسيق البيانات المرسلة للنموذج
    input_data = [image, data]
    
    # تطبيق التنبؤ باستخدام النموذج على البيانات
    predictions = model.predict(input_data)
    
    # تحويل النتيجة إلى توقع فئة
    predicted_class = np.argmax(predictions)
    
    # تحويل التوقع إلى تسمية فئة
    label = label_encoder.inverse_transform([predicted_class])[0]
    
    # إعداد النتيجة
    result = {
        'prediction': label
    }
    
    return result

In [45]:
# تحميل الصورة وتحويلها إلى نصف قطري محدد (224x224 بكسل مثلاً)
image = Image.open('/kaggle/input/test-1/download (1).jfif')  # استبدال 'path_to_image.jpg' بمسار الصورة الفعلي
image = image.resize((224, 224))  # تحديد الحجم المطلوب للصورة

# تحويل الصورة إلى مصفوفة NumPy
image_array = np.array(image)

# قم بتطبيق أي عمليات معالجة مسبقة إضافية إذا لزم الأمر (مثل تقليص القيم إلى النطاق [0, 1])

# تنفيذ التنبؤ باستخدام النموذج
predictions = model.predict(np.expand_dims(image_array, axis=0))

# تحويل النتائج إلى تنبؤ فئة أو قيمة محددة
predicted_class = np.argmax(predictions)

print('Predicted Class:', predicted_class)

ValueError: Exception encountered when calling Sequential.call().

[1mInput 0 of layer "dense" is incompatible with the layer: expected axis -1 of input shape to have value 28800, but received input with shape (1, 200704)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(1, 224, 224, 3), dtype=uint8)
  • training=False
  • mask=None